# PE Head-Only — Single Model (SEED=52)

Trains one `PE_model_h52.pth` and generates `submission_h52.csv`.

## 1. Imports & Device

In [ ]:
import os
import random
import timm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from torchvision.transforms import v2 as T
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

SEED = 52
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
print(f'Seed: {SEED}')

## 2. Data

In [ ]:
DATA_DIR = '.'

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomCrop(336),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomGrayscale(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def make_dataset(transform):
    ds = datasets.ImageFolder(os.path.join(DATA_DIR, 'train'), transform=transform)
    ds.class_to_idx = {cls: int(cls) for cls in ds.classes}
    ds.samples = [(path, int(ds.classes[lbl])) for path, lbl in ds.samples]
    ds.targets  = [int(ds.classes[lbl]) for lbl in ds.targets]
    return ds

_base = make_dataset(train_transform)
val_size   = int(0.1 * len(_base))
train_size = len(_base) - val_size
train_idx, val_idx = random_split(
    range(len(_base)), [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_dataset = Subset(make_dataset(train_transform), train_idx.indices)
val_dataset   = Subset(make_dataset(eval_transform),  val_idx.indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=0)

print(f'Total: {len(_base)} | Train: {len(train_dataset)} | Val: {len(val_dataset)}')

## 3. Model

In [ ]:
NUM_CLASSES = 100

class PEHeadClassifier(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.backbone = timm.create_model(
            'vit_pe_core_large_patch14_336.fb',
            pretrained=True,
            num_classes=0
        )
        for param in self.backbone.parameters():
            param.requires_grad = False
        self.head = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(self.backbone.num_features, num_classes)
        )

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        return self.head(features)

model = PEHeadClassifier(NUM_CLASSES).to(device)
print(f'Feature dim: {model.backbone.num_features}')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 4. Train

In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
    return total_loss / total, correct / total

criterion = nn.CrossEntropyLoss()
cutmix    = T.CutMix(num_classes=NUM_CLASSES)

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = cutmix(images, labels)
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels.argmax(1)).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total

EPOCHS    = 50
optimizer = optim.Adam(model.head.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

train_losses, val_losses, train_accs, val_accs = [], [], [], []
best_val_acc = 0.0

print('=== Training (backbone frozen) ===')
for epoch in range(EPOCHS):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion)
    scheduler.step()
    train_losses.append(tr_loss); train_accs.append(tr_acc)
    val_losses.append(vl_loss);   val_accs.append(vl_acc)
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {tr_loss:.3f} Acc: {tr_acc:.3f} | Val Loss: {vl_loss:.3f} Acc: {vl_acc:.3f}')
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), 'PE_model_h52.pth')
        print(f'  -> Saved! Val acc: {vl_acc:.3f}')

print(f'\nBest val acc: {best_val_acc:.3f}')

## 5. Training Curves

In [ ]:
epochs_range = range(1, EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, train_losses, label='Train')
ax1.plot(epochs_range, val_losses,   label='Val')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss'); ax1.legend()

ax2.plot(epochs_range, train_accs, label='Train')
ax2.plot(epochs_range, val_accs,   label='Val')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Accuracy'); ax2.legend()

plt.suptitle(f'PE Head-Only SEED={SEED} | Best Val: {best_val_acc:.3f}')
plt.tight_layout()
plt.savefig('pe_h52_training_curves.png', dpi=150)
plt.show()

## 6. Submission (single model + TTA)

In [7]:
TTA_RUNS = 5

tta_transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.RandomCrop(336),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

test_dir   = os.path.join(DATA_DIR, 'test')
test_files = sorted(os.listdir(test_dir), key=lambda x: int(x.split('.')[0]))

model.load_state_dict(torch.load('PE_model_h52.pth', map_location=device))
model.eval()

all_probs = []
with torch.no_grad():
    for fname in test_files:
        img = Image.open(os.path.join(test_dir, fname)).convert('RGB')
        probs = F.softmax(model(eval_transform(img).unsqueeze(0).to(device)), dim=1)
        for _ in range(TTA_RUNS):
            probs += F.softmax(model(tta_transform(img).unsqueeze(0).to(device)), dim=1)
        probs /= (TTA_RUNS + 1)
        all_probs.append(probs.cpu().numpy())

pe_probs = np.array(all_probs)
np.save('pe_probs_h52.npy', pe_probs)
print(f'Saved pe_probs_h52.npy with shape {pe_probs.shape}')

/tmp/ipykernel_324/1892879215.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('PE_model_h52.pth', map_location=device))


Saved pe_probs_h52.npy with shape (1036, 1, 100)
